# 05. Unpacking and keyword-only

**Tier:** Intermediate. **Why you'd use this:** A `move` helper must take `x` and `y` in that order, and an optional `label` that callers should pass by name so a number cannot land in the wrong slot.

**After this lesson you can:** mark keyword-only and positional-only parameters; unpack a list into arguments with `*`; unpack a dict into keywords with `**`.
About 25 minutes. **Uses:** [Arguments](02-arguments-lesson.ipynb)


## Quick review

1. What does `*args` collect inside a definition?
2. Are type hints checked when the function runs?

<details><summary>Answers</summary>

Extra positional arguments, as a tuple. No. Hints are for readers and tools.
</details>


## The idea

In a definition, `/` ends the positional-only parameters. A bare `*` starts the keyword-only parameters. Callers must pass those by name. At a call site, `*` unpacks a sequence into positional arguments, and `**` unpacks a dict into keyword arguments.

<details><summary>Explain more</summary>

`def move(x, y, /, *, label="pt")` means `x` and `y` cannot be passed by name, and `label` cannot be passed by position. `move(*point)` is `move(point[0], point[1])` when `point` has two items. `move(**{"x": 1, "y": 2})` fails here because `x` and `y` are positional-only.
</details>


## See it work

Move a point with a required pair and a keyword-only label.


In [1]:
def move(x, y, /, *, label="pt"):
    # 1. x and y are positional-only.
    # 2. label must be passed by name when given.
    return label + ":" + str(x) + "," + str(y)


point = [3, 4]
print(move(1, 2))
print(move(*point, label="A"))


pt:1,2
A:3,4


## Check yourself

Answer these before you run the cell.

1. Which call fails, and why?
2. What does `pair(1, 2)` print when `options` supplies `scale`?


In [2]:
def move(x, y, /, *, label="pt"):
    return label + ":" + str(x) + "," + str(y)


print(move(1, 2, "A"))


TypeError: move() takes 2 positional arguments but 3 were given

In [3]:
def pair(a, b, *, scale=1):
    return (a * scale, b * scale)


options = {"scale": 10}
print(pair(1, 2, **options))


(10, 20)


<details><summary>Answers</summary>

`move(1, 2, "A")` fails because `label` is keyword-only. `pair(1, 2, **options)` prints `(10, 20)`.
</details>


## Why it works this way

`/` and `*` are contracts in the signature. Positional-only parameters keep a short left-to-right API stable when you rename internals. Callers never depended on the old names. Keyword-only parameters stop a bare value from sliding into the wrong slot. The call `move(1, 2, "A")` is a `TypeError` because the third positional argument has nowhere to go. That failure is the point: a silent swap of label and coordinate would be worse. Deep dive: [Scopes and closures](../../internals/06-scopes/01-closures-lesson.ipynb).

`*` and `**` at the call site iterate the container and fill arguments. The cost is one pass over that container plus the call. `*` on a long list still passes one argument per item. It does not pass the list as a single object. `**` matches keys to parameter names. A leftover key is an error. A missing required name is an error. The same star syntax packs in a definition and unpacks at a call, so the mental model stays one idea. `co_posonlyargcount` reports how many leading parameters the `/` locked. Putting the restriction in the signature means every call site sees it. A comment that says "pass label by name" will not raise.

Call-site unpacking is how you adapt data you already have. A two-item point becomes `move(*point)` without renaming locals to `x` and `y`. A dict of options becomes `pair(**options)` without spelling every key. The contract still holds: positional-only names cannot arrive through `**`, and keyword-only names cannot arrive through a bare extra positional value.


In [4]:
def move(x, y, /, *, label="pt"):
    return label + ":" + str(x) + "," + str(y)


print(move.__code__.co_posonlyargcount)
print(move(0, 0, label="origin"))


2
origin:0,0


## When to use it

Use keyword-only for options that are easy to swap by accident. Use positional-only for short numeric APIs you want to rename later. Use `*` and `**` at the call site when values already sit in a list or dict. Don't unpack a dict into positional-only names. Pass those with `*`.


## Common mistake

Passing a keyword-only option by position. The third value looks like a label and raises instead.


In [5]:
def move(x, y, /, *, label="pt"):
    return label + ":" + str(x) + "," + str(y)


print(move(1, 2, "A"))


TypeError: move() takes 2 positional arguments but 3 were given

Name the option. The `/` and `*` in the signature are what make that required.


In [6]:
def move(x, y, /, *, label="pt"):
    return label + ":" + str(x) + "," + str(y)


print(move(1, 2, label="A"))


A:1,2


## Try it

Change `move` so `label` has no default and must be supplied. What error do you get from `move(1, 2)`, and why is that better than silently using `"pt"` for a real dataset?


## Recap

- `/` marks positional-only parameters. `*` marks keyword-only ones.
- At a call, `*` unpacks a sequence and `**` unpacks a dict.
- Those marks catch swapped arguments that a plain signature would accept.

**Next:** [practice](05-unpacking-and-keyword-only-practice.ipynb)
